# AI-IO on T-Lab: Run all
**Steps:** Runtime → Run all. Allow Google Drive access when asked (one click). Nothing else to edit.

Pre-trains AI-IO on EuRoC (25 epochs, checkpoint `0024.ckpt`), fine-tunes it on the three T-Lab splits (`exp0_july_only`, `exp1_sepoct_20pct`, `exp2_sepoct_40pct`) and prints the 9 / 12 / 25 s horizon results. Code: [`souvik0306/Air-IO`](https://github.com/souvik0306/Air-IO) (`adaptive_loss_function`) + [`Mndarish/AI-IO-TLab`](https://github.com/Mndarish/AI-IO-TLab).

- **Needs a GPU runtime.** The notebook is set to GPU; if Colab gave you CPU, use Runtime → Change runtime type → T4.
- **Everything is saved to Drive** in `MyDrive/AIIO_colab` (checkpoints, results, logs). If Colab disconnects or runs out of GPU quota, reconnect and **Run all again**: it resumes from the last finished epoch and skips finished phases.
- **Weights & Biases (optional):** add a Colab secret named `WANDB_API_KEY` (key icon in the left bar) to log online. Without it, runs log offline to `MyDrive/AIIO_colab/wandb`; upload later with `wandb sync`.
- **Private repo?** Add a Colab secret `GITHUB_TOKEN` with read access to `Mndarish/AI-IO-TLab`.

In [ ]:
# ===== Settings (no need to change) =====
import os
DRIVE_DIR      = "/content/drive/MyDrive/AIIO_colab"
REPO           = "/content/Air-IO"
OVERLAY        = "/content/AI-IO-TLab"
BASE_URL       = "https://github.com/souvik0306/Air-IO.git"
BASE_BRANCH    = "adaptive_loss_function"
OVERLAY_URL    = "https://github.com/Mndarish/AI-IO-TLab.git"
OVERLAY_BRANCH = "main"
EPOCHS_PRE     = 25          # EuRoC pre-training epochs; fine-tuning starts from the epoch-24 checkpoint (0024.ckpt)
EPOCHS_FT      = 30
EXPS           = ["exp0_july_only", "exp1_sepoct_20pct", "exp2_sepoct_40pct"]

# W&B: online if a WANDB_API_KEY secret exists, otherwise offline (no prompt)
key = None
try:
    from google.colab import userdata; key = userdata.get("WANDB_API_KEY")
except Exception: pass
WANDB_MODE = "online" if key else "offline"
if key: os.environ["WANDB_API_KEY"] = key
os.environ.update(AIIO_EPOCHS_PRE=str(EPOCHS_PRE), AIIO_EPOCHS_FT=str(EPOCHS_FT), AIIO_WANDB=WANDB_MODE,
                  WANDB_MODE=WANDB_MODE, AIIO_WANDB_CHECK="0", AIRIO_REPO_DIR=REPO, PBS_O_WORKDIR=REPO)
print("W&B mode:", WANDB_MODE)

In [ ]:
# ===== Drive + GPU =====
import subprocess, pathlib, shutil, glob
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{DRIVE_DIR}/done", exist_ok=True)

def sh(cmd, cwd=None, env=None, check=True):
    print("\n$", cmd)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or (REPO if os.path.isdir(REPO) else None), env=env or os.environ,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, executable="/bin/bash")
    for line in p.stdout: print(line, end="")
    rc = p.wait()
    if check and rc != 0: raise RuntimeError(f"failed (exit {rc}): {cmd}")
    return rc

assert subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode == 0, \
    "No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again."
sh("nvidia-smi --query-gpu=name,memory.total --format=csv", cwd="/content")

In [ ]:
# ===== Code: clone base repo + overlay, copy overlay on top, patch =====
tok = None
try:
    from google.colab import userdata; tok = userdata.get("GITHUB_TOKEN")
except Exception: pass
ov_url = OVERLAY_URL.replace("https://", f"https://{tok}@") if tok else OVERLAY_URL

if not os.path.isdir(REPO):
    sh(f"git clone -q --branch {BASE_BRANCH} {BASE_URL} {REPO}", cwd="/content")
if not os.path.isdir(OVERLAY):
    sh(f"git clone -q --branch {OVERLAY_BRANCH} {ov_url} {OVERLAY}", cwd="/content")
else:
    sh("git pull -q", cwd=OVERLAY)
sh("git log --oneline -1", cwd=REPO); sh("git log --oneline -1", cwd=OVERLAY)

shutil.copytree(OVERLAY, REPO, ignore=shutil.ignore_patterns(".git", "colab", "docs", "README.md", ".gitignore"), dirs_exist_ok=True)
sh("chmod +x *.sh tools/*.sh")
sh(f"python3 {OVERLAY}/colab/apply_patches.py {REPO}")

In [ ]:
# ===== Environment: shims for the Vanda scripts + python deps =====
sh(f"chmod +x {OVERLAY}/colab/shims/*", cwd="/content")
os.environ["PATH"] = f"{OVERLAY}/colab/shims:" + os.environ["PATH"]
keep = [l for l in open(f"{REPO}/requirements.txt").read().split()
        if not l.startswith(("torch==", "numpy==", "scipy==", "matplotlib==", "opencv"))]
open("/content/requirements-colab.txt", "w").write("\n".join(keep) + "\n")
sh("pip install -q -r /content/requirements-colab.txt")

for d in ["experiments", "result", "results", "results_csv", "orientations", "wandb"]:
    os.makedirs(f"{DRIVE_DIR}/{d}", exist_ok=True)
    p = f"{REPO}/{d}"
    if os.path.islink(p): continue
    if os.path.exists(p): shutil.rmtree(p)
    os.symlink(f"{DRIVE_DIR}/{d}", p)

In [ ]:
# ===== Data (T-Lab + EuRoC), AI-IO configs, self-test =====
sh("bash setup_aiio.sh 2>&1 | tail -40")
hits = glob.glob(f"{REPO}/data/euroc/**/MH_01_easy", recursive=True)
assert hits, "EuRoC not downloaded"
link = f"{REPO}/EuRoC-Dataset"
if os.path.lexists(link): os.remove(link)
os.symlink(os.path.dirname(hits[0]), link)

In [ ]:
# ===== Runner =====
def run_phase(name, script, extra_env=None):
    marker = pathlib.Path(f"{DRIVE_DIR}/done/{name}")
    if marker.exists():
        print(f"[skip] {name} already finished"); return
    env = dict(os.environ, **(extra_env or {}))
    env.update(WANDB_RUN_ID=f"aiio_{name}", WANDB_RESUME="allow")
    print(f"[run] {name}")
    sh(f"bash {script}", cwd=REPO, env=env)
    marker.touch(); print(f"[done] {name}")

In [ ]:
# ===== 1. EuRoC pre-training (resumes if interrupted) =====
run_phase("pretrain", "aiio_pretrain.pbs")

In [ ]:
# ===== 2. Fine-tune + inference + metrics + horizons, per split =====
cdir = f"{REPO}/experiments/aiio_euroc/aiio_tf/ckpt"
ck = f"{cdir}/0024.ckpt"
if not os.path.isfile(ck):
    ck = f"{cdir}/best_model.ckpt"; print("0024.ckpt not found, using best_model.ckpt")
assert os.path.isfile(ck), os.listdir(cdir)
os.environ["AIIO_CHECKPOINT"] = ck
print("fine-tuning from", ck)
for e in EXPS:
    run_phase(f"exp_{e}", "aiio_exp.pbs", {"AIIO_EXP": e})

In [ ]:
# ===== 3. Results (9 / 12 / 25 s, net-only) =====
sh("python3 tools/summarize_horizons.py --mode net --by_group")
sh("python3 tools/compare_horizons.py")
print("\nAll outputs are in", DRIVE_DIR)